In [ ]:
"""
================================================================================
LEVEL 3 - TASK 1: RANDOM FOREST CLASSIFIER
================================================================================
Objective: Build an ensemble model for improved classification
Dataset: Iris Dataset
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("="*80)
print("LEVEL 3 - TASK 1: RANDOM FOREST CLASSIFIER")
print("="*80)

# ============================================================================
# STEP 1: SETUP PATHS
# ============================================================================

current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
main_dir = os.path.dirname(current_dir)
dataset_path = os.path.join(main_dir, 'datasets', '1) iris.csv')
output_dir = os.path.join(main_dir, 'outputs')
image_dir = os.path.join(main_dir, 'images')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(image_dir, exist_ok=True)

print(f"📁 Dataset path: {dataset_path}")

# ============================================================================
# STEP 2: LOAD AND PREPARE DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 2: LOAD AND PREPARE DATA")
print("="*80)

# Load dataset
if os.path.exists(dataset_path):
    iris_df = pd.read_csv(dataset_path)
    print(f"✅ Loaded from: {dataset_path}")
else:
    alt_paths = ['../datasets/1) iris.csv', './datasets/1) iris.csv', '1) iris.csv']
    for path in alt_paths:
        if os.path.exists(path):
            iris_df = pd.read_csv(path)
            print(f"✅ Loaded from: {path}")
            break

# Encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(iris_df['species'])

# Features
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris_df[features]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📌 DATASET: {X.shape[0]} samples, {X.shape[1]} features")
print(f"📌 Classes: {label_encoder.classes_}")
print(f"📌 Training: {X_train.shape[0]} samples")
print(f"📌 Testing: {X_test.shape[0]} samples")

# ============================================================================
# STEP 3: TRAIN RANDOM FOREST WITH DIFFERENT N_ESTIMATORS
# ============================================================================

print("\n" + "="*80)
print("STEP 3: TRAIN WITH DIFFERENT N_ESTIMATORS")
print("="*80)

n_estimators_list = [10, 50, 100, 150, 200]
rf_scores = []

print("\n📌 Testing different numbers of trees:")
print("-" * 60)
print(f"{'Trees':<10} {'Train Acc':<12} {'Test Acc':<12} {'CV Mean':<12}")
print("-" * 60)

for n in n_estimators_list:
    rf = RandomForestClassifier(n_estimators=n, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    
    train_acc = rf.score(X_train, y_train)
    test_acc = rf.score(X_test, y_test)
    cv_scores = cross_val_score(rf, X_train, y_train, cv=5)
    cv_mean = cv_scores.mean()
    rf_scores.append({'n_estimators': n, 'train_acc': train_acc, 'test_acc': test_acc, 'cv_mean': cv_mean})
    
    print(f"{n:<10} {train_acc:.4f}      {test_acc:.4f}      {cv_mean:.4f}")

# Find best
best_idx = np.argmax([s['test_acc'] for s in rf_scores])
best_n = rf_scores[best_idx]['n_estimators']
print(f"\n🏆 Best number of trees: {best_n}")

# ============================================================================
# STEP 4: TRAIN BEST RANDOM FOREST
# ============================================================================

print("\n" + "="*80)
print("STEP 4: TRAIN BEST RANDOM FOREST")
print("="*80)

rf_best = RandomForestClassifier(n_estimators=best_n, max_depth=5, random_state=42)
rf_best.fit(X_train, y_train)
y_pred = rf_best.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"✅ Best Random Forest trained!")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • Number of trees: {best_n}")

# ============================================================================
# STEP 5: CROSS-VALIDATION
# ============================================================================

print("\n" + "="*80)
print("STEP 5: CROSS-VALIDATION")
print("="*80)

cv_scores = cross_val_score(rf_best, X_train, y_train, cv=5)
print(f"📌 5-Fold Cross-Validation Scores: {cv_scores}")
print(f"📌 Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# ============================================================================
# STEP 6: FEATURE IMPORTANCE
# ============================================================================

print("\n" + "="*80)
print("STEP 6: FEATURE IMPORTANCE")
print("="*80)

feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf_best.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n📌 FEATURE IMPORTANCE:")
print(feature_importance)

# ============================================================================
# STEP 7: EVALUATE
# ============================================================================

print("\n" + "="*80)
print("STEP 7: EVALUATE")
print("="*80)

print("\n📌 CONFUSION MATRIX:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n📌 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# ============================================================================
# STEP 8: COMPARE WITH DECISION TREE
# ============================================================================

print("\n" + "="*80)
print("STEP 8: COMPARE WITH DECISION TREE")
print("="*80)

from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_accuracy = dt.score(X_test, y_test)

print(f"📌 Random Forest Accuracy: {accuracy:.4f}")
print(f"📌 Decision Tree Accuracy: {dt_accuracy:.4f}")
print(f"📌 Improvement: {(accuracy - dt_accuracy)*100:.2f}%")

# ============================================================================
# STEP 9: VISUALIZE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 9: VISUALIZE RESULTS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
axes[0,0].set_title('Confusion Matrix')

# 2. Feature Importance
axes[0,1].barh(feature_importance['Feature'], feature_importance['Importance'], color='green')
axes[0,1].set_xlabel('Importance')
axes[0,1].set_title('Feature Importance')
axes[0,1].grid(True, alpha=0.3, axis='x')
axes[0,1].invert_yaxis()

# 3. Number of Trees vs Accuracy
n_values = [s['n_estimators'] for s in rf_scores]
test_accs = [s['test_acc'] for s in rf_scores]
cv_means = [s['cv_mean'] for s in rf_scores]

axes[1,0].plot(n_values, test_accs, 'b-o', label='Test Accuracy', linewidth=2)
axes[1,0].plot(n_values, cv_means, 'r-s', label='CV Mean', linewidth=2)
axes[1,0].set_xlabel('Number of Trees')
axes[1,0].set_ylabel('Accuracy')
axes[1,0].set_title('Random Forest Performance')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 4. Model Comparison
models = ['Random Forest', 'Decision Tree']
scores = [accuracy, dt_accuracy]
bars = axes[1,1].bar(models, scores, color=['green', 'orange'])
axes[1,1].set_ylabel('Accuracy')
axes[1,1].set_title('Model Comparison')
axes[1,1].set_ylim(0.8, 1.0)
for bar, score in zip(bars, scores):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.01,
                   f'{score:.4f}', ha='center', va='bottom')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(image_dir, 'random_forest_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(image_dir, 'random_forest_results.png')}")

# ============================================================================
# STEP 10: SAVE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 10: SAVE RESULTS")
print("="*80)

results_df = pd.DataFrame({
    'Actual': label_encoder.inverse_transform(y_test),
    'Predicted': label_encoder.inverse_transform(y_pred),
    'Correct': y_test == y_pred
})
results_df.to_csv(os.path.join(output_dir, 'random_forest_results.csv'), index=False)
print(f"✅ Saved: {os.path.join(output_dir, 'random_forest_results.csv')}")

# ============================================================================
# STEP 11: SUMMARY
# ============================================================================

print("\n" + "="*80)
print("STEP 11: SUMMARY")
print("="*80)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    LEVEL 3 - TASK 1 COMPLETED                             ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  ✅ Random Forest Classifier                                              ║
║                                                                            ║
║  PERFORMANCE:                                                             ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • Test Accuracy: {accuracy:.4f}                                          ║
║  • CV Mean: {cv_scores.mean():.4f}                                        ║
║  • Best Trees: {best_n}                                                   ║
║  • Improvement over DT: {(accuracy - dt_accuracy)*100:.2f}%               ║
║                                                                            ║
║  OUTPUT FILES:                                                            ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • outputs/random_forest_results.csv                                      ║
║  • images/random_forest_results.png                                       ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

print("\n🎉 RANDOM FOREST COMPLETED SUCCESSFULLY!")